# Tokenización y Vectorización en NLP y LLMs

### Conceptos, aplicaciones y ejemplos prácticos

**Inteligencia Artificial — Dr. Jorge Roa · Dra. Milagros Gutiérrez**

*CIDISI — UTN — FRSF*

Abril 2026

## El mapa del curso

| Clase | Tema | Pregunta central |
|---|---|---|
| **1** | NLP, Tokenización, Embeddings | ¿Cómo representa texto una máquina? |
| 2 | LLMs | ¿Cómo genera texto un modelo? |
| 3 | RAG | ¿Cómo le doy conocimiento propio? |
| 4 | Agentes | ¿Cómo actúa de forma autónoma? |

🎯 **TP integrador:** agente RAG funcional sobre un dominio a elección

## El pipeline de un LLM

```
  Texto         ✂️ Tokenización    🧭 Embeddings      ⚙️ Modelo              💬 Output
  de entrada  ─────────────────────────────────────────────────────────────────────────▶
  (hoy)           (hoy)              (hoy)             (Clase siguiente)     (Clase siguiente)
```

## Objetivos de la clase

- Entender qué es la tokenización y vectorización y por qué son fundamentales en NLP y LLMs.
- Conocer el concepto de word embeddings y su uso.
- Aprender cómo se vectorizan los prompts.
- Comprender la similaridad de vectores y su aplicación.
- Ver ejemplos prácticos.
- Introducción a los vector stores (bases de datos vectoriales).

**Al final de la clase vas a poder explicar cómo los modelos de lenguaje entienden semánticamente el texto.**

## El problema de fondo — tokenización

*¿Por qué necesitamos tokenizar el texto?*

- **Vocabulario infinito:** palabras nuevas, jerga, errores, nombres propios, hashtags.
- **Costo computacional:** un vocabulario por palabras tendría millones de entradas — matrices de embeddings inmanejables.
- **Morfología rica del español:** "jugar / jugando / jugaste" comparten raíz pero serían tokens distintos.
- **Idiomas sin separadores:** chino, japonés, tailandés no usan espacios entre palabras.

→ La **tokenización** descompone el texto en piezas manejables sin que se rompa con palabras nuevas o idiomas distintos.

## El problema de fondo — embeddings

**Pregunta de fondo: ¿cómo le damos semántica a las palabras?**

Una computadora ve `"auto"` y `"sandía"` como cadenas opacas — no sabe qué significan. De ahí salen tres síntomas:

- **Sinonimia:** "auto" y "coche" son palabras distintas pero equivalentes.
- **Polisemia:** "banco" puede ser institución o mueble.
- **Contexto:** "el banco quebró" vs "me senté en el banco".

→ Los **embeddings** resuelven esto haciendo que el significado emerja de la **geometría**: palabras similares = vectores cercanos.

## ¿Qué es la tokenización?

- La **tokenización** es el proceso de dividir un texto en unidades más pequeñas llamadas **tokens**.
- Un token puede ser una palabra, subpalabra, carácter o símbolo.
- Ejemplo: `"El perro duerme"` → `["El", "perro", "duerme"]`

## ¿Por qué es necesaria la tokenización?

- Los modelos de IA no pueden trabajar directamente con texto crudo.
- La tokenización transforma el texto en una secuencia de unidades manejables.
- Permite manejar palabras desconocidas, errores de ortografía y diferentes idiomas.
- Es el primer paso antes de convertir el texto en vectores (embeddings).

## Tipos de tokenización

**Existen diferentes formas de dividir el texto en tokens:**

- **Tokenización por caracteres:**
  - Cada letra o símbolo es un token.
  - Ejemplo: `"gato"` → `["g", "a", "t", "o"]`
  - *Ventaja:* máxima flexibilidad, nunca hay palabras desconocidas.
  - *Desventaja:* secuencias muy largas, pierde información semántica.
- **Tokenización por palabras:**
  - Cada palabra es un token.
  - Ejemplo: `"El perro juega"` → `["El", "perro", "juega"]`
  - *Ventaja:* simple y fácil de entender.
  - *Desventaja:* no maneja bien palabras desconocidas o errores de ortografía.
- **Tokenización por subpalabras:**
  - Las palabras se dividen en partes más pequeñas (subpalabras o *chunks*).
  - Ejemplo: `"jugando"` → `["jug", "ando"]`
  - *Ventaja:* permite manejar palabras nuevas, conjugaciones y errores.
  - *Desventaja:* los tokens pueden no tener significado por sí solos.

## La subpalabra ganó — ¿cómo se aprende el vocabulario?

**Todos los LLMs modernos tokenizan por subpalabras.** Por palabras o caracteres ya casi no se usa en producción.

### La idea común a todos los algoritmos:

1. Arrancás con un vocabulario de **caracteres** sueltos.
2. Recorrés el corpus y **fusionás el par de tokens más frecuente** repetidamente.
3. Parás cuando alcanzás el tamaño de vocabulario objetivo (típicamente 30K – 200K tokens).

```
Corpus: "chatear" y "ChatGPT" aparecen muchas veces
    ↓ pares frecuentes: "ch", "at", "Ch", "GPT"
    ↓ tras varias fusiones:
"ChatGPT"  →  ["Chat", "GPT"]
"chatear"  →  ["chat", "ear"]
```

→ Aunque "ChatGPT" nunca esté en el vocabulario como palabra entera, **sí lo están sus partes**. El modelo nunca queda mudo.

## BPE, WordPiece, SentencePiece

**Los tres aprenden vocabulario de subpalabras. La diferencia está en *cómo eligen qué par fusionar*.**

| Algoritmo | Modelos | Ejemplo |
|---|---|---|
| **BPE** | GPT-4o, Llama, Mistral | `token` + `ización` |
| **WordPiece** | BERT, DistilBERT | `token` + `##ización` |
| **SentencePiece** | Llama 2, T5, mT5 | `▁token` + `ización` |

**BPE** — fusiona el par de caracteres más frecuente iterativamente  
**WordPiece** — maximiza la probabilidad del corpus (prefija subpalabras con `##`)  
**SentencePiece** — no pre-tokeniza por espacios, ideal para idiomas sin separadores

## Variantes modernas (opcional)

Más allá de los tres algoritmos clásicos, en producción se usan algunas variantes:

- **Tiktoken** (OpenAI) — implementación propia de BPE.  
  Tokenizers: `cl100k_base` (GPT-4), `o200k_base` (GPT-4o, GPT-4.1).

- **Byte-level BPE** — opera sobre **bytes** en vez de caracteres Unicode.  
  Ventaja: nunca falla con ningún carácter (emojis, símbolos raros, código).  
  Lo usan: GPT-2 y descendientes, Llama, Mistral.

- **Unigram** (parte de SentencePiece) — alternativa a BPE.  
  En vez de fusiones greedy, usa un modelo probabilístico que asigna probabilidades a candidatos de subpalabras y elige los más probables.  
  Lo usan: T5, mT5, ALBERT.

→ Todos comparten la misma idea de fondo: **vocabulario de subpalabras aprendido del corpus**.

## Algoritmo general — pseudocódigo

```
ENTRADA:
    corpus            # texto plano de entrenamiento
    vocab_size        # tamaño objetivo (ej: 50000)

ALGORITMO:

    1. vocab ← {todos los caracteres únicos del corpus}

    2. tokenizar el corpus en caracteres
       ej: "chat"  →  ["c", "h", "a", "t"]

    3. mientras |vocab| < vocab_size:

         a. contar todos los pares adyacentes en el corpus tokenizado
            ej: ("c","h"): 8420 veces
                ("h","a"): 4310 veces
                ("a","t"): 2100 veces
                ...

         b. mejor_par ← el par más frecuente

         c. fusionar mejor_par en todo el corpus
            "c h a t"  →  "ch a t"

         d. agregar "ch" al vocabulario

    4. retornar vocab + reglas de fusión

SALIDA:
    vocab           # mapa token → id
    reglas_fusión   # se aplican en el mismo orden al inferir
```

→ Es exactamente lo que hace **BPE**.  
→ **WordPiece** cambia el paso (b): elige el par que más aumenta la *probabilidad del corpus*, no el más frecuente.  
→ **Unigram** funciona al revés: arranca con vocabulario grande y va *quitando* tokens que aportan poco.

## Tokenización en inferencia

Una vez entrenado el tokenizer, el vocabulario y las reglas de fusión quedan **fijas**.
Al tokenizar una palabra nueva, se aplican las reglas en el orden aprendido y se prefieren los tokens más largos:

```
"chatear"      →  ["chat", "ear"]              (2 tokens)
"chateando"    →  ["chat", "e", "ando"]         (3 tokens)
"chatáfono"    →  ["ch", "at", "á", "f", "o", "no"]  (palabra inventada → más tokens, más chicos)
```

→ **Palabras frecuentes** se descomponen en pocas piezas grandes.  
→ **Palabras raras o inventadas** se descomponen en muchas piezas chicas.  
→ Como `"ch"` quedó en el vocabulario desde la iteración 1, sigue disponible para palabras donde no se pueda usar un token más largo.

Por eso el costo de un prompt en una API (medido en tokens) **depende del idioma y del vocabulario del modelo** — no del número de palabras.

## Quiz de tokenización

*¿Verdadero o falso?*

1. La tokenización por subpalabras es la que usan todos los LLMs modernos en producción. <span class="fragment">**Verdadero**</span>

2. Tokenizar significa convertir directamente el texto en vectores numéricos. <span class="fragment">**Falso**</span>

3. Una palabra que el modelo nunca vio durante el entrenamiento del tokenizer puede igual ser tokenizada, descomponiéndola en partes más pequeñas. <span class="fragment">**Verdadero**</span>

4. Cada palabra del español ocupa un token en GPT-4o. <span class="fragment">**Falso**</span>

5. Una vez que un token entra al vocabulario durante el entrenamiento, queda ahí para siempre. <span class="fragment">**Verdadero**</span>

---

- Ejercitación: [Ir a Notebook](https://colab.research.google.com/drive/1LL-4FD0pkpl4zpbWYREUuspsDYNg0P3u#scrollTo=0TRVc0xrzRWW)

## ¿Qué es la vectorización?

- **Vectorizar** es transformar datos (por ejemplo, texto) en vectores numéricos.
- Los modelos de IA sólo pueden trabajar con números, no con palabras o frases.
- **Ejemplo:** la palabra "gato" podría representarse como el vector `[0.3, 0.1, 0.65]`.
- **Analogía:** es como traducir un idioma humano a un idioma que la computadora entiende.

## ¿Por qué es importante en NLP y LLMs?

- El texto es información no estructurada y ambigua.
- La vectorización permite:
  - Procesar texto con modelos matemáticos.
  - Comparar y buscar similitudes entre textos.
  - Aprender relaciones semánticas y sintácticas.
- Sin vectorización, los LLMs no podrían "entender" ni generar texto.

## Ejemplo motivador: de texto a números

**¿Cómo representamos la frase "El perro juega"?**

- **One-hot encoding:** cada palabra es un vector con un 1 en la posición correspondiente y 0 en las demás.
- **Problema:** no captura relaciones entre palabras, y los vectores son muy grandes y dispersos.

*Ejemplo: "perro" = `[0, 1, 0, 0, 0, ...]`*

## Limitaciones del one-hot encoding

- No refleja similitud semántica: "perro" y "gato" son igual de diferentes que "perro" y "auto".
- Los vectores son muy grandes si el vocabulario es grande.
- No permite generalizar ni aprender relaciones entre palabras.

**Solución:** representaciones que pesan palabras por relevancia (BoW, TF-IDF) y, más adelante, **embeddings** que capturan significado.

## Bag of Words — primera mejora sobre one-hot

**Idea:** representar cada **documento** como un vector de conteos de palabras.

```
Doc 1: "El gato come pescado"
Doc 2: "El perro come carne"
Doc 3: "El gato duerme"

        carne  come  duerme  el  gato  perro  pescado
Doc 1:    0      1      0     1    1      0       1
Doc 2:    1      1      0     1    0      1       0
Doc 3:    0      0      1     1    1      0       0
```

⚠️ **Problema:** "auto" y "coche" siguen siendo columnas distintas sin ninguna relación.  
⚠️ **Problema:** el orden de las palabras se pierde completamente.

> 📖 *Jurafsky, D. & Martin, J. H. (2024). Speech and Language Processing, Cap. 6. https://web.stanford.edu/~jurafsky/slp3/6.pdf*

## TF-IDF — pesando palabras por relevancia

**Problema de BoW:** trata todas las palabras igual. "el", "la", "de" reciben el mismo peso que "tokenización".

**Idea:** una palabra que aparece en **todos** los documentos dice poco. Una que aparece en **uno solo** dice mucho.

```
  TF-IDF(término, documento) = TF × IDF

  TF  = frecuencia del término en el documento
        (cuántas veces aparece "gato" en Doc 1)

  IDF = log( N / cantidad de docs que contienen el término )
        (si "el" aparece en los 3 docs: IDF bajo → peso bajo)
        (si "pescado" aparece en 1 de 3: IDF alto → peso alto)
```

| Término | Aparece en | TF-IDF |
|---|---|---|
| "el" | Todos los docs | **muy bajo** — no distingue |
| "come" | 2 de 3 docs | **bajo** |
| "gato" | 2 de 3 docs | **medio** |
| "pescado" | 1 de 3 docs | **alto** — distingue Doc 1 |

> 📖 *Jurafsky, D. & Martin, J. H. (2024). Speech and Language Processing, Cap. 6. https://web.stanford.edu/~jurafsky/slp3/6.pdf*

## Word embeddings: ¿qué son y para qué sirven?

- Son representaciones densas y continuas de palabras en un espacio vectorial de baja dimensión (por ejemplo, 300 dimensiones).
- Capturan relaciones semánticas y sintácticas entre palabras.
- Palabras con significados similares tienen vectores cercanos.
- Ejemplo: $\text{king} - \text{man} + \text{woman} \approx \text{queen}$
- Modelos populares: Word2Vec, GloVe, FastText, embeddings de LLMs.

*Los embeddings se pueden aprender a partir de grandes corpus de texto.*

## Embeddings — representar significado como vector

**¿Recuerdan el problema de BoW?** "auto" y "coche" eran columnas distintas sin relación.

Los embeddings lo resuelven:

- **Palabras con contextos similares → vectores cercanos**  
  "auto" y "coche" aparecen cerca de "manejar", "rueda", "velocidad" → sus vectores quedan cerca

- **La distancia codifica significado**  
  "auto" está lejos de "sandía" — no tienen nada en común semánticamente

- **Word2Vec — limitación ⚠️**  
  Cada palabra tiene UN solo vector: `"banco"` (dinero) = `"banco"` (mueble) → mismo punto

> 📖 *Mikolov, T., et al. (2013). Efficient Estimation of Word Representations in Vector Space. https://arxiv.org/abs/1301.3781*

## Visualización de word embeddings

![Visualización de embeddings](figures/embeddings.png)

- Palabras con significados similares están cerca en el espacio vectorial.
- Relaciones como género, tiempo verbal, etc. se pueden representar como "direcciones" en el espacio.

## ¿Cómo se obtienen los word embeddings?

- Se entrenan modelos para que palabras que aparecen en contextos similares tengan vectores similares.
- **Ejemplo:** Word2Vec predice palabras vecinas en una oración.
- Los LLMs generan embeddings contextuales: el vector de una palabra depende de la frase.
- Visualización: [https://dash.gallery/dash-word-arithmetic/](https://dash.gallery/dash-word-arithmetic/)

*Hoy en día, los embeddings de LLMs son el estándar para tareas avanzadas.*

## Sentence Transformers — del word al sentence embedding

Los word embeddings clásicos (Word2Vec, GloVe) tienen dos limitaciones:

- **Un vector por palabra**, sin importar el contexto.  
  → "banco" (dinero) y "banco" (mueble) caen en el mismo punto.
- **Solo representan palabras sueltas**, no frases ni oraciones.  
  ¿Cómo represento *"El gato duerme en la cocina"* como un único vector?

Los **Sentence Transformers** resuelven ambos problemas:

- Generan **embeddings contextuales**: el vector de una palabra depende del resto de la frase.
- Devuelven un **único vector por oración** — listo para usar en búsqueda semántica, clasificación, RAG.
- Se construyen sobre arquitecturas Transformer entrenadas con pares de oraciones similares (Siamese BERT).

> 📖 *Reimers, N. & Gurevych, I. (2019). Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks. EMNLP 2019. https://arxiv.org/abs/1908.10084*

## Modelos de sentence embeddings

**Open source (corren local, gratis):**

- `paraphrase-multilingual-MiniLM-L12-v2` — liviano (~90 MB), buen español
- `all-MiniLM-L6-v2` — el más usado en inglés
- `BAAI/bge-*` — top de benchmarks actuales (MTEB)
- `intfloat/multilingual-e5-*` — fuerte en multilingüe

**APIs comerciales (cerrados, con costo):**

- OpenAI `text-embedding-3-small` / `text-embedding-3-large`
- Cohere `embed-multilingual-v3.0`
- Google `text-embedding-004`

→ De acá en adelante, cuando digamos "embedding" nos referimos a un **sentence embedding**.

> 🔗 Benchmark: [MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard)

## Tokenización y embeddings

### Pipeline de 3 pasos:

1. **Tokenización:** el texto se divide en tokens.
2. **Mapeo a IDs:** cada token se convierte en un número entero (ID).
3. **Embeddings:** cada ID se transforma en un vector numérico, que el LLM puede procesar.

→ La búsqueda de vector por ID se hace contra una **tabla de embeddings** entrenada junto con el modelo. Cada token tiene su vector fijo allí.

## Vectorización de prompts

- Un **prompt** es el texto de entrada que le damos a un LLM.
- El modelo tokeniza el prompt (lo divide en palabras o subpalabras).
- Cada token se convierte en un vector (embedding).
- El LLM procesa la secuencia de vectores para generar una respuesta.

## Ejemplo: vectorización de un prompt

**Prompt:** *¿Cuál es la capital de Argentina?*

- Tokenización: `["¿", "Cuál", "es", "la", "capital", "de", "Argentina", "?"]`
- Cada token se transforma en un vector numérico, por ejemplo:
  - "capital" → `[0.12, 0.98, -0.33, ...]`
  - "Argentina" → `[0.45, -0.22, 0.67, ...]`
- El LLM procesa la secuencia de estos vectores para entender el significado del prompt.

## Métricas de similitud — coseno

**Mide:** el ángulo entre vectores. Ignora la magnitud.

$$\text{sim}(A, B) = \frac{A \cdot B}{\|A\| \, \|B\|}$$

- **Rango:** −1 a 1. En la práctica con embeddings: 0 a 1.
- **Cuándo se usa:** **default en NLP**.

Sentence Transformers, OpenAI embeddings, y los vector stores (FAISS, Chroma, Pinecone) la usan por defecto.

## Métricas de similitud — producto punto

**Mide:** similitud teniendo en cuenta magnitud + ángulo.

$$A \cdot B = \sum_i a_i \, b_i$$

- **Rango:** −∞ a +∞ (sin acotar).
- **Cuándo se usa:** si los embeddings están **normalizados** (norma = 1), el producto punto es **exactamente igual** a la similitud coseno — pero más rápido de calcular.

Por eso muchos modelos (OpenAI, BGE, E5) ya devuelven embeddings normalizados, y vector stores como FAISS usan producto punto internamente.

## Métricas de similitud — distancia euclidiana (L2)

**Mide:** la distancia "en línea recta" entre dos puntos del espacio vectorial.

$$d(A, B) = \|A - B\| = \sqrt{\sum_i (a_i - b_i)^2}$$

- **Rango:** 0 (vectores idénticos) a +∞.
- **Cuándo se usa:** menos común para texto. Aparece en:
  - **Visualizaciones** — UMAP, t-SNE.
  - **Clustering** — k-means.
  - **Recuperación de imágenes**.

→ A diferencia del coseno, sí depende de la magnitud — por eso en NLP suele perder.

## ¿Qué es un vector store?

- Es una base de datos optimizada para almacenar y buscar vectores.
- Permite búsquedas semánticas: encontrar textos similares a un prompt.
- Ejemplos: FAISS, ChromaDB, Milvus, Pinecone.
- Muy usado en sistemas de retrieval-augmented generation (RAG) con LLMs.
- **Ventaja:** permite búsquedas rápidas en millones de documentos.

## Ejemplo de uso de vector store

- **Caso:** buscador de preguntas frecuentes (FAQ).
- Se vectorizan todas las preguntas y se almacenan en un vector store.
- Cuando un usuario hace una pregunta, se vectoriza el prompt y se busca la pregunta más similar.
- **Ventaja:** encuentra respuestas aunque la pregunta no sea exactamente igual.

## Aplicaciones de la vectorización

- Búsqueda semántica de documentos: encontrar textos relevantes aunque no coincidan exactamente las palabras.
- Clasificación de texto: spam, sentimiento, temas, etc.
- Detección de similitud y plagio.
- Sistemas de recomendación de contenido.
- Retrieval-augmented generation (RAG) en LLMs: combinar búsqueda y generación de texto.
- Chatbots inteligentes y asistentes virtuales.

## Ejercitación

- Vectorización manual: [Ir a Notebook](https://colab.research.google.com/drive/1lWR99IhnOiHOnKlOUWuQlBL8pcHY5PD5#scrollTo=Fx-0HLDUK_wM)
- Vectorización automática: [Ir a Notebook](https://colab.research.google.com/drive/10Ds4ZQdVTWCi_ZxqktKeMYL76MM1uiXm#scrollTo=0HSh_TkDow-W)
- Ejemplo con base de datos vectorial: [Ir a Notebook](https://colab.research.google.com/drive/1-gbWhC5NyxAoB8Rd9KQm8k4eGlmw4GKL#scrollTo=FR7XgQ_RwSY1)

## Resumen

- La vectorización es clave para que los modelos de NLP y LLMs procesen texto.
- Los word embeddings permiten representar palabras y textos en forma numérica y semántica.
- La similitud de vectores es fundamental para comparar textos y buscar información relevante.
- Los vector stores permiten búsquedas eficientes y semánticas en grandes volúmenes de datos.

**¡La vectorización es el puente entre el lenguaje humano y la inteligencia artificial!**

## 📚 Bibliografía

### Libros de referencia

- Jurafsky, D. & Martin, J. H. (2024). *Speech and Language Processing* (3rd ed. draft). Stanford University.  
  🔗 https://web.stanford.edu/~jurafsky/slp3/

- Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*. MIT Press.  
  🔗 https://www.deeplearningbook.org/

- Tunstall, L., von Werra, L. & Wolf, T. (2022). *Natural Language Processing with Transformers*. O'Reilly Media.  
  🔗 https://www.oreilly.com/library/view/natural-language-processing/9781098136789/

### Papers

- Mikolov, T., Chen, K., Corrado, G. & Dean, J. (2013). Efficient Estimation of Word Representations in Vector Space. *ICLR 2013*.  
  🔗 https://arxiv.org/abs/1301.3781

- Sennrich, R., Haddow, B. & Birch, A. (2016). Neural Machine Translation of Rare Words with Subword Units. *ACL 2016*.  
  🔗 https://arxiv.org/abs/1508.07909

- Devlin, J., Chang, M., Lee, K. & Toutanova, K. (2018). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. *NAACL 2019*.  
  🔗 https://arxiv.org/abs/1810.04805

- Reimers, N. & Gurevych, I. (2019). Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks. *EMNLP 2019*.  
  🔗 https://arxiv.org/abs/1908.10084

- McInnes, L., Healy, J. & Melville, J. (2018). UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction.  
  🔗 https://arxiv.org/abs/1802.03426